In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Set up directories
base_dir = Path.cwd().parent
raw_data_dir = base_dir / 'data' / 'raw'
processed_data_dir = base_dir / 'data' / 'processed'
processed_data_dir.mkdir(parents=True, exist_ok=True)

print("Data Preprocessing Pipeline Initialized")
print("Raw data path:", raw_data_dir)
print("Processed data path:", processed_data_dir)

Data Preprocessing Pipeline Initialized
Raw data path: c:\Users\induw\OneDrive\Documents\SENTIO_360_Project\data\raw
Processed data path: c:\Users\induw\OneDrive\Documents\SENTIO_360_Project\data\processed


In [2]:
print("Processing Text Data...")

# 1. LOAD & CLEAN WAF ATTACKS DATA
waf_path = raw_data_dir / 'waf_attacks' / 'payload_full.csv'
df_waf = pd.read_csv(waf_path)
df_waf = df_waf[['payload', 'attack_type']].dropna()

def categorize_waf(attack_type):
    if attack_type == 'norm':
        return 'Safe'
    return 'Web_Attack'

df_waf['label'] = df_waf['attack_type'].apply(categorize_waf)
df_waf.rename(columns={'payload': 'text'}, inplace=True)
# Data Science Standard: Removing duplicates to prevent Data Leakage
df_waf_clean = df_waf[['text', 'label']].drop_duplicates()


# 2. LOAD & CLEAN LLM PROMPTS DATA
llm_dir = raw_data_dir / 'llm_prompts'
llm_train_file = list(llm_dir.glob('train*.parquet'))[0]
llm_test_file = list(llm_dir.glob('test*.parquet'))[0]

df_llm_train = pd.read_parquet(llm_train_file)
df_llm_test = pd.read_parquet(llm_test_file)
df_llm = pd.concat([df_llm_train, df_llm_test], ignore_index=True)

df_llm['label'] = df_llm['label'].apply(lambda x: 'LLM_Attack' if x == 1 else 'Safe')
df_llm_clean = df_llm[['text', 'label']].dropna().drop_duplicates()


# 3. LOAD & CLEAN SAFE TEXT DATA 
safe_train_path = raw_data_dir / 'safe_text' / 'train.csv'
safe_test_path = raw_data_dir / 'safe_text' / 'test.csv'

df_safe_train = pd.read_csv(safe_train_path, names=['class', 'title', 'description'], header=0)
df_safe_test = pd.read_csv(safe_test_path, names=['class', 'title', 'description'], header=0)
df_safe = pd.concat([df_safe_train, df_safe_test], ignore_index=True)

df_safe['text'] = df_safe['title'] + " " + df_safe['description']
df_safe['label'] = 'Safe'
df_safe_clean = df_safe[['text', 'label']].dropna().drop_duplicates()

# Balance the dataset (Undersampling Safe text to match attack ratios)
df_safe_clean = df_safe_clean.sample(n=50000, random_state=42)


# 4. MERGE & SHUFFLE
df_text_master = pd.concat([df_waf_clean, df_llm_clean, df_safe_clean], ignore_index=True)
df_text_master = df_text_master.sample(frac=1, random_state=42).reset_index(drop=True)

df_text_master.to_csv(processed_data_dir / 'master_text_data.csv', index=False)
print(f"Text data preprocessing complete. Total Records: {len(df_text_master)}")

Processing Text Data...
Text data preprocessing complete. Total Records: 81729


In [4]:
print("\n--- 2. Processing Network Behavior Data ---")

network_path = raw_data_dir / 'network_traffic' / 'cicids2017.csv'

# Load the original large dataset
print("Loading original network dataset...")
df_network = pd.read_csv(network_path, low_memory=False)

# Clean column names (removing extra spaces)
df_network.columns = df_network.columns.str.strip()

# Handle infinite values (caused by dividing by zero in network packets) and Missing values
print("Cleaning Data (Removing Inf, NaN, and Duplicates)...")
df_network.replace([np.inf, -np.inf], np.nan, inplace=True)
df_network = df_network.dropna().drop_duplicates()

# MANUAL FEATURE SELECTION BASED ON EDA (Not AI selected)
# Selected these features because Correlation Heatmap and Boxplots 
# showed they have the highest variance between Safe and Attack traffic.
selected_features = [
    'Max Packet Length', 'Packet Length Variance', 'Avg Bwd Segment Size', 
    'Average Packet Size', 'Bwd Packet Length Max', 'Destination Port', 
    'Packet Length Std', 'Total Length of Bwd Packets', 'Subflow Fwd Bytes', 
    'Bwd Header Length', 'Label'
]

# Keep only the selected features
available_features = [col for col in selected_features if col in df_network.columns]
df_final = df_network[available_features].copy()

# Standardize Target Variable
df_final['label'] = df_final['Label'].apply(lambda x: 'Safe' if str(x).strip() == 'BENIGN' else 'Network_Attack')
df_final.drop(columns=['Label'], inplace=True)

# Balance Dataset to avoid Model Bias
print("Balancing the final dataset (50k Safe, 50k Attack)...")
df_safe_net = df_final[df_final['label'] == 'Safe'].sample(n=50000, random_state=42)

# Ensure we don't sample more attacks than available
num_attacks = len(df_final[df_final['label'] == 'Network_Attack'])
attack_sample_size = min(50000, num_attacks)
df_attack_net = df_final[df_final['label'] == 'Network_Attack'].sample(n=attack_sample_size, random_state=42)

# Merge and shuffle
master_network_df = pd.concat([df_safe_net, df_attack_net]).sample(frac=1, random_state=42).reset_index(drop=True)

# Export to processed folder
net_export_path = processed_data_dir / 'master_behavior_data.csv'
master_network_df.to_csv(net_export_path, index=False)

print(f"[SUCCESS] Master Behavior Dataset saved successfully.")
print(f"[INFO] Final Behavior Dataset Shape: {master_network_df.shape}")
print(f"[INFO] Class Distribution:\n{master_network_df['label'].value_counts()}")


--- 2. Processing Network Behavior Data ---
Loading original network dataset...
Cleaning Data (Removing Inf, NaN, and Duplicates)...
Balancing the final dataset (50k Safe, 50k Attack)...
[SUCCESS] Master Behavior Dataset saved successfully.
[INFO] Final Behavior Dataset Shape: (100000, 11)
[INFO] Class Distribution:
label
Network_Attack    50000
Safe              50000
Name: count, dtype: int64


In [5]:
from PIL import Image

print("\n--- 3. Processing Vision Data Mapping ---")

# 1. Gather Malware Images
malware_dir = raw_data_dir / 'malware_images'
malware_images = list(malware_dir.rglob('*.png')) + list(malware_dir.rglob('*.jpg'))

vision_data = []
for img_path in malware_images:
    vision_data.append({
        'image_path': str(img_path.resolve()), 
        'label': 'Malware_Image'
    })

num_malware = len(vision_data)
print(f"Found {num_malware} Malware Images. Mapping paths to memory...")

# 2. Generate 'Safe' Texture Images to Balance the Dataset
print(f"Generating {num_malware} 'Safe' Texture Images from Benign Text Bytes...")
safe_images_dir = processed_data_dir / 'safe_textures'
safe_images_dir.mkdir(parents=True, exist_ok=True)

# Load Safe text from our recently processed master text dataset
text_df = pd.read_csv(processed_data_dir / 'master_text_data.csv')
safe_texts = text_df[text_df['label'] == 'Safe']['text'].dropna().head(num_malware).tolist()

for i, text in enumerate(safe_texts):
    # Convert text to bytes
    bytes_data = text.encode('utf-8', errors='ignore')
    
    # Standardize length: 64x64 image requires exactly 4096 bytes
    if len(bytes_data) < 4096:
        bytes_data = bytes_data.ljust(4096, b'\x00') # Pad with null bytes if too short
    else:
        bytes_data = bytes_data[:4096] # Truncate if too long
        
    # Convert bytes to 64x64 pixel array (0-255 values)
    img_array = np.frombuffer(bytes_data, dtype=np.uint8).reshape((64, 64))
    
    # Save as Grayscale Image
    img = Image.fromarray(img_array, 'L')
    img_name = safe_images_dir / f'safe_texture_{i}.png'
    img.save(img_name)
    
    # Map the generated safe image path
    vision_data.append({
        'image_path': str(img_name.resolve()), 
        'label': 'Safe_Image'
    })

# 3. Export Master Vision Dataset Mapping
df_vision = pd.DataFrame(vision_data)
# Shuffle the dataset to prevent sequence bias during CNN training
df_vision = df_vision.sample(frac=1, random_state=42).reset_index(drop=True)

vision_export_path = processed_data_dir / 'master_vision_data.csv'
df_vision.to_csv(vision_export_path, index=False)

print(f"[SUCCESS] Master Vision Mapping saved successfully.")
print(f"[INFO] Final Vision Dataset Shape: {df_vision.shape}")
print(f"[INFO] Class Distribution:\n{df_vision['label'].value_counts()}")
print("\nDATA PREPROCESSING PIPELINE FINISHED SUCCESSFULLY.")


--- 3. Processing Vision Data Mapping ---
Found 9339 Malware Images. Mapping paths to memory...
Generating 9339 'Safe' Texture Images from Benign Text Bytes...
[SUCCESS] Master Vision Mapping saved successfully.
[INFO] Final Vision Dataset Shape: (18678, 2)
[INFO] Class Distribution:
label
Malware_Image    9339
Safe_Image       9339
Name: count, dtype: int64

DATA PREPROCESSING PIPELINE FINISHED SUCCESSFULLY.
